# Expected Loss Modeling

## Objective

The goal of this notebook is to estimate the expected claim cost for an Auto insurance contract.

Expected Loss combines two components:

- claim occurrence probability,
- expected claim severity.

The basic formula is:

Expected Loss = Claim Probability × Expected Severity

The claim occurrence model did not produce reliable individual probabilities on the final test set.

Therefore, the training portfolio claim rate will be used as the initial frequency benchmark.

Claim severity models that use `claim_type` cannot be used directly before a claim occurs because claim type is not known at contract inception.

For this reason, the Expected Loss analysis will use only information available before claim occurrence.

# Expected Loss Modeling

## Objective

The goal of this notebook is to estimate the expected claim cost for an Auto insurance contract.

Expected Loss combines two components:

- claim occurrence probability,
- expected claim severity.

The basic formula is:

Expected Loss = Claim Probability × Expected Severity

The claim occurrence model did not produce reliable individual probabilities on the final test set.

Therefore, the training portfolio claim rate will be used as the initial frequency benchmark.

Claim severity models that use `claim_type` cannot be used directly before a claim occurs because claim type is not known at contract inception.

For this reason, the Expected Loss analysis will use only information available before claim occurrence.

In [31]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyodbc

from sklearn.model_selection import RepeatedStratifiedKFold

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [2]:
conn = pyodbc.connect(
    "DSN=InsuranceAnalytics;",
    autocommit=True,
)

In [3]:
occurrence_query = """
SELECT *
FROM vw_auto_claim_occurrence_ml
"""

occurrence_df = pd.read_sql(
    occurrence_query,
    conn,
)

print("Occurrence shape:", occurrence_df.shape)
print(
    "Unique contracts:",
    occurrence_df["contract_id"].nunique(),
)
print(
    "Claims:",
    occurrence_df["has_claim"].sum(),
)

Occurrence shape: (4327, 20)
Unique contracts: 4327
Claims: 88


In [4]:
split_path = Path(
    "../data/processed/occurrence_split_map.csv"
)

split_df = pd.read_csv(
    split_path,
    dtype={"contract_id": "string"},
)

print("Split shape:", split_df.shape)

display(
    split_df["dataset"].value_counts()
)

Split shape: (4327, 2)


dataset
Train    3461
Test      866
Name: count, dtype: int64

In [5]:
expected_loss_df = occurrence_df.merge(
    split_df,
    on="contract_id",
    how="left",
    validate="one_to_one",
)

In [6]:
print(
    "Missing split labels:",
    expected_loss_df["dataset"].isna().sum(),
)

split_check = (
    expected_loss_df
    .groupby("dataset")
    .agg(
        contracts=("contract_id", "size"),
        claims=("has_claim", "sum"),
        claim_rate=("has_claim", "mean"),
    )
)

split_check["claim_rate_pct"] = (
    split_check["claim_rate"] * 100
)

split_check

Missing split labels: 0


,contracts,claims,claim_rate,claim_rate_pct
dataset,,,,
Test,866,18,0.0208,2.0785
Train,3461,70,0.0202,2.0225


In [7]:
severity_query = """
SELECT
    contract_id,
    claim_id,
    damage_amount
FROM vw_auto_claim_severity_ml
"""

severity_df = pd.read_sql(
    severity_query,
    conn,
)

print("Severity claims:", len(severity_df))
print(
    "Unique contracts:",
    severity_df["contract_id"].nunique(),
)

Severity claims: 109
Unique contracts: 109


In [8]:
severity_split = severity_df.merge(
    split_df,
    on="contract_id",
    how="inner",
    validate="one_to_one",
)

severity_split_summary = (
    severity_split
    .groupby("dataset")
    .agg(
        claims=("claim_id", "size"),
        mean_damage=("damage_amount", "mean"),
        median_damage=("damage_amount", "median"),
        total_damage=("damage_amount", "sum"),
    )
)

severity_split_summary

,claims,mean_damage,median_damage,total_damage
dataset,,,,
Test,18,"5,008.7961","2,768.7550","90,158.3300"
Train,70,"4,538.9144","2,068.2150","317,724.0100"


## Portfolio Expected Loss Benchmark

The initial Expected Loss benchmark is estimated using only the training data.

The frequency component is the training claim rate.

The severity component is the mean damage amount among training claims.

Expected Loss is calculated as:

Expected Loss = Claim Frequency × Mean Claim Severity

In [9]:
train_population = expected_loss_df[
    expected_loss_df["dataset"] == "Train"
]

train_claims = severity_split[
    severity_split["dataset"] == "Train"
]

train_claim_rate = (
    train_population["has_claim"].mean()
)

train_mean_severity = (
    train_claims["damage_amount"].mean()
)

portfolio_expected_loss = (
    train_claim_rate
    * train_mean_severity
)

print(
    "Train claim rate:",
    round(train_claim_rate, 6),
)

print(
    "Train mean severity:",
    round(train_mean_severity, 2),
)

print(
    "Portfolio Expected Loss:",
    round(portfolio_expected_loss, 2),
)

Train claim rate: 0.020225
Train mean severity: 4538.91
Portfolio Expected Loss: 91.8


In [11]:
test_population = expected_loss_df[
    expected_loss_df["dataset"] == "Test"
].copy()

test_population[
    "predicted_expected_loss"
] = portfolio_expected_loss

In [12]:
test_population.head()

,contract_id,start_date,end_date,annual_premium,city,risk_zone,client_age,channel,csp,gender,brand,model,year,power_hp,fuel_type,current_value,color,vehicle_usage,previous_claims,has_claim,dataset,predicted_expected_loss
1,CTR_000012,2024-03-16,2025-04-20,878.3000,Marseille,Medium,41.0000,Phone,Worker,NaN,Peugeot,208,"2,020.0000",175.0000,Hybrid,"8,362.3000",White,Professional,2.0000,0,Test,91.8012
4,CTR_000018,2023-09-19,2024-08-05,732.3000,Bordeaux,Medium,36.0000,Web,NaN,NaN,Renault,Megane,"2,018.0000",163.6000,Diesel,"5,563.7300",Blue,Personal,NaN,0,Test,91.8012
9,CTR_000031,2024-08-16,2025-07-11,531.7100,Nantes,Low,62.0000,Phone,Worker,Male,Renault,Captur,"2,010.0000",NaN,Diesel,"5,874.5800",Gray,Mixed,1.0000,0,Test,91.8012
11,CTR_000035,2023-03-17,2024-03-05,543.6700,Dijon,Low,36.0000,Web,Worker,NaN,Peugeot,3008,"2,012.0000",183.0000,Electric,"4,635.4700",Black,Professional,0.0000,0,Test,91.8012
17,CTR_000054,2023-04-17,2024-03-31,936.3100,Marseille,Medium,41.0000,Web,Worker,Male,Renault,Megane,"2,019.0000",95.6700,Electric,"5,836.7500",Red,Personal,1.0000,0,Test,91.8012


In [14]:
test_actual_loss = (
    test_population[
        ["contract_id"]
    ]
    .merge(
        severity_split.loc[
            severity_split["dataset"] == "Test",
            [
                "contract_id",
                "damage_amount",
            ],
        ],
        on="contract_id",
        how="left",
    )
)

test_actual_loss["actual_loss"] = (
    test_actual_loss["damage_amount"]
    .fillna(0)
)

In [17]:
expected_loss_evaluation = pd.Series(
    {
        "test_contracts": len(test_actual_loss),

        "predicted_avg_loss": (
            portfolio_expected_loss
        ),

        "actual_avg_loss": (
            test_actual_loss[
                "actual_loss"
            ].mean()
        ),

        "predicted_total_loss": (
            portfolio_expected_loss
            * len(test_actual_loss)
        ),

        "actual_total_loss": (
            test_actual_loss[
                "actual_loss"
            ].sum()
        ),
    }
)

expected_loss_evaluation

test_contracts            866.0000
predicted_avg_loss         91.8012
actual_avg_loss           104.1089
predicted_total_loss   79,499.8534
actual_total_loss      90,158.3300
dtype: float64

### Portfolio Benchmark Evaluation

The training-based Expected Loss benchmark slightly underestimates the test portfolio loss.

The predicted average loss per contract is about 91.8, while the actual average loss is about 104.1.

At portfolio level, the benchmark underestimates the realized total loss by about 11.8%.

Considering the small number of claims in the test population, some variation between predicted and realized losses is expected.

In [18]:
predicted_total_loss = (
    expected_loss_evaluation["predicted_total_loss"]
)

actual_total_loss = (
    expected_loss_evaluation["actual_total_loss"]
)

loss_error = (
    predicted_total_loss
    - actual_total_loss
)

absolute_error = abs(loss_error)

percentage_error = (
    loss_error
    / actual_total_loss
    * 100
)

absolute_percentage_error = abs(
    percentage_error
)

loss_ratio_actual_to_expected = (
    actual_total_loss
    / predicted_total_loss
)

evaluation_metrics = pd.Series(
    {
        "predicted_total_loss": predicted_total_loss,
        "actual_total_loss": actual_total_loss,
        "error": loss_error,
        "absolute_error": absolute_error,
        "percentage_error_pct": percentage_error,
        "absolute_percentage_error_pct": (
            absolute_percentage_error
        ),
        "actual_to_expected_ratio": (
            loss_ratio_actual_to_expected
        ),
    }
)

evaluation_metrics

predicted_total_loss             79,499.8534
actual_total_loss                90,158.3300
error                           -10,658.4766
absolute_error                   10,658.4766
percentage_error_pct                -11.8220
absolute_percentage_error_pct        11.8220
actual_to_expected_ratio              1.1341
dtype: float64

## Risk-Zone Expected Loss Benchmark

A single portfolio-level Expected Loss gives the same prediction to every contract.

As a simple improvement, Expected Loss is estimated separately for each risk zone.

For each risk zone, only training data is used to estimate:

- claim frequency,
- mean claim severity,
- expected loss.

Expected Loss = Claim Frequency × Mean Claim Severity

In [19]:
loss_df = expected_loss_df.merge(
    severity_split[
        [
            "contract_id",
            "damage_amount",
        ]
    ],
    on="contract_id",
    how="left",
    validate="one_to_one",
)

loss_df["actual_loss"] = (
    loss_df["damage_amount"]
    .fillna(0)
)

In [20]:
train_loss_df = loss_df[
    loss_df["dataset"] == "Train"
].copy()

risk_zone_frequency = (
    train_loss_df
    .groupby("risk_zone")
    .agg(
        contracts=("contract_id", "size"),
        claims=("has_claim", "sum"),
    )
)

risk_zone_frequency["claim_rate"] = (
    risk_zone_frequency["claims"]
    / risk_zone_frequency["contracts"]
)

In [22]:
risk_zone_severity = (
    train_loss_df.loc[
        train_loss_df["has_claim"] == 1
    ]
    .groupby("risk_zone")
    .agg(
        mean_severity=("actual_loss", "mean")
    )
)

In [23]:
risk_zone_el = (
    risk_zone_frequency
    .join(risk_zone_severity)
)

risk_zone_el["expected_loss"] = (
    risk_zone_el["claim_rate"]
    * risk_zone_el["mean_severity"]
)

risk_zone_el

,contracts,claims,claim_rate,mean_severity,expected_loss
risk_zone,,,,,
High,962,23,0.0239,"3,886.6035",92.9230
Low,1020,23,0.0225,"5,234.7726",118.0390
Medium,1479,24,0.0162,"4,497.1817",72.9766


In [24]:
test_loss_df = loss_df[
    loss_df["dataset"] == "Test"
].copy()

test_loss_df["predicted_el_risk_zone"] = (
    test_loss_df["risk_zone"]
    .map(risk_zone_el["expected_loss"])
)

In [25]:
test_loss_df["predicted_el_risk_zone"] = (
    test_loss_df["predicted_el_risk_zone"]
    .fillna(portfolio_expected_loss)
)

In [26]:
risk_zone_predicted_total = (
    test_loss_df["predicted_el_risk_zone"].sum()
)

actual_total = (
    test_loss_df["actual_loss"].sum()
)

risk_zone_error = (
    risk_zone_predicted_total
    - actual_total
)

risk_zone_pct_error = (
    risk_zone_error
    / actual_total
    * 100
)

risk_zone_evaluation = pd.Series(
    {
        "predicted_avg_loss": (
            test_loss_df[
                "predicted_el_risk_zone"
            ].mean()
        ),

        "actual_avg_loss": (
            test_loss_df["actual_loss"].mean()
        ),

        "predicted_total_loss": (
            risk_zone_predicted_total
        ),

        "actual_total_loss": (
            actual_total
        ),

        "percentage_error_pct": (
            risk_zone_pct_error
        ),

        "absolute_percentage_error_pct": (
            abs(risk_zone_pct_error)
        ),
    }
)

risk_zone_evaluation

predicted_avg_loss                  92.1846
actual_avg_loss                    104.1089
predicted_total_loss            79,831.8237
actual_total_loss               90,158.3300
percentage_error_pct               -11.4537
absolute_percentage_error_pct       11.4537
dtype: float64

In [27]:
benchmark_comparison = pd.DataFrame(
    {
        "model": [
            "Portfolio Benchmark",
            "Risk Zone Benchmark",
        ],

        "predicted_total_loss": [
            predicted_total_loss,
            risk_zone_predicted_total,
        ],

        "actual_total_loss": [
            actual_total,
            actual_total,
        ],

        "absolute_percentage_error_pct": [
            absolute_percentage_error,
            abs(risk_zone_pct_error),
        ],
    }
)

benchmark_comparison

,model,predicted_total_loss,actual_total_loss,absolute_percentage_error_pct
0,Portfolio Benchmark,"79,499.8534","90,158.3300",11.8220
1,Risk Zone Benchmark,"79,831.8237","90,158.3300",11.4537


In [28]:
risk_zone_test_evaluation = (
    test_loss_df
    .groupby("risk_zone")
    .agg(
        contracts=("contract_id", "size"),
        claims=("has_claim", "sum"),
        predicted_total_loss=(
            "predicted_el_risk_zone",
            "sum",
        ),
        actual_total_loss=(
            "actual_loss",
            "sum",
        ),
    )
)

risk_zone_test_evaluation[
    "predicted_avg_loss"
] = (
    risk_zone_test_evaluation[
        "predicted_total_loss"
    ]
    / risk_zone_test_evaluation["contracts"]
)

risk_zone_test_evaluation[
    "actual_avg_loss"
] = (
    risk_zone_test_evaluation[
        "actual_total_loss"
    ]
    / risk_zone_test_evaluation["contracts"]
)

risk_zone_test_evaluation[
    "error"
] = (
    risk_zone_test_evaluation[
        "predicted_total_loss"
    ]
    - risk_zone_test_evaluation[
        "actual_total_loss"
    ]
)

risk_zone_test_evaluation[
    "percentage_error_pct"
] = (
    risk_zone_test_evaluation["error"]
    / risk_zone_test_evaluation["actual_total_loss"]
    * 100
)

risk_zone_test_evaluation

,contracts,claims,predicted_total_loss,actual_total_loss,predicted_avg_loss,actual_avg_loss,error,percentage_error_pct
risk_zone,,,,,,,,
High,233,3,"21,651.0479","9,873.3700",92.9230,42.3750,"11,777.6779",119.2873
Low,266,5,"31,398.3714","38,634.5400",118.0390,145.2426,"-7,236.1686",-18.7298
Medium,367,10,"26,782.4044","41,650.4200",72.9766,113.4889,"-14,868.0156",-35.6972


In [29]:
test_frequency = (
    test_loss_df
    .groupby("risk_zone")
    .agg(
        contracts=("contract_id", "size"),
        claims=("has_claim", "sum"),
    )
)

test_frequency["claim_rate"] = (
    test_frequency["claims"]
    / test_frequency["contracts"]
)

test_severity = (
    test_loss_df.loc[
        test_loss_df["has_claim"] == 1
    ]
    .groupby("risk_zone")
    .agg(
        mean_severity=(
            "actual_loss",
            "mean",
        )
    )
)

test_risk_zone_actuals = (
    test_frequency
    .join(test_severity)
)

test_risk_zone_actuals["actual_expected_loss"] = (
    test_risk_zone_actuals["claim_rate"]
    * test_risk_zone_actuals["mean_severity"]
)

test_risk_zone_actuals

,contracts,claims,claim_rate,mean_severity,actual_expected_loss
risk_zone,,,,,
High,233,3,0.0129,"3,291.1233",42.3750
Low,266,5,0.0188,"7,726.9080",145.2426
Medium,367,10,0.0272,"4,165.0420",113.4889


### Risk Zone Benchmark Result

Risk-zone segmentation provides only a very small improvement in total portfolio error.

However, the segment-level predictions are unstable.

The High-risk segment is substantially overestimated, while the Low and Medium segments are underestimated.

Therefore, the small portfolio-level improvement appears to come from offsetting segment errors rather than stable risk differentiation.

Risk zone will not replace the portfolio-level Expected Loss benchmark.

In [30]:
train_cv_df = train_loss_df.copy()

train_cv_df["vehicle_age"] = (
    pd.to_datetime(train_cv_df["start_date"]).dt.year
    - train_cv_df["year"]
).clip(lower=0)

train_cv_df["vehicle_age_group"] = pd.cut(
    train_cv_df["vehicle_age"],
    bins=[-1, 3, 7, np.inf],
    labels=["0-3", "4-7", "8+"],
).astype("object")

train_cv_df["client_age_group"] = pd.cut(
    train_cv_df["client_age"],
    bins=[17, 29, 39, 49, 59, np.inf],
    labels=[
        "18-29",
        "30-39",
        "40-49",
        "50-59",
        "60+",
    ],
).astype("object")

train_cv_df["previous_claims_cat"] = (
    train_cv_df["previous_claims"]
    .astype("Int64")
    .astype("string")
    .fillna("Unknown")
)

## Train-Only Expected Loss Validation

Several simple segmentation strategies are compared using only the training population.

For every cross-validation fold:

1. claim frequency and mean severity are estimated from the fold's training data,
2. segment-level Expected Loss is calculated,
3. the Expected Loss values are applied to the validation contracts,
4. predicted portfolio loss is compared with realized portfolio loss.

The test population is not used for selecting the segmentation strategy.

In [32]:
el_cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

In [33]:
def calculate_segment_el(
    train_df,
    segment_col,
):
    frequency = (
        train_df
        .groupby(segment_col, dropna=False)
        .agg(
            contracts=("contract_id", "size"),
            claims=("has_claim", "sum"),
        )
    )

    frequency["claim_rate"] = (
        frequency["claims"]
        / frequency["contracts"]
    )

    severity = (
        train_df.loc[
            train_df["has_claim"] == 1
        ]
        .groupby(segment_col, dropna=False)
        .agg(
            mean_severity=("actual_loss", "mean")
        )
    )

    result = frequency.join(severity)

    result["expected_loss"] = (
        result["claim_rate"]
        * result["mean_severity"]
    )

    return result

In [34]:
segment_candidates = [
    "risk_zone",
    "vehicle_age_group",
    "client_age_group",
    "previous_claims_cat",
    "channel",
]

In [35]:
el_cv_results = []

X_dummy = train_cv_df.drop(
    columns="has_claim"
)

y_cv = train_cv_df["has_claim"]

for fold, (cv_train_idx, cv_val_idx) in enumerate(
    el_cv.split(X_dummy, y_cv),
    start=1,
):
    fold_train = train_cv_df.iloc[
        cv_train_idx
    ].copy()

    fold_val = train_cv_df.iloc[
        cv_val_idx
    ].copy()

    # -----------------------------
    # Portfolio benchmark
    # -----------------------------
    fold_claim_rate = (
        fold_train["has_claim"].mean()
    )

    fold_mean_severity = (
        fold_train.loc[
            fold_train["has_claim"] == 1,
            "actual_loss",
        ].mean()
    )

    portfolio_el = (
        fold_claim_rate
        * fold_mean_severity
    )

    portfolio_predicted_total = (
        portfolio_el * len(fold_val)
    )

    actual_total = (
        fold_val["actual_loss"].sum()
    )

    portfolio_ape = (
        abs(
            portfolio_predicted_total
            - actual_total
        )
        / actual_total
        * 100
    )

    el_cv_results.append(
        {
            "model": "Portfolio",
            "fold": fold,
            "predicted_total": (
                portfolio_predicted_total
            ),
            "actual_total": actual_total,
            "absolute_percentage_error": (
                portfolio_ape
            ),
        }
    )

    # -----------------------------
    # Segment benchmarks
    # -----------------------------
    for segment_col in segment_candidates:

        segment_el = calculate_segment_el(
            fold_train,
            segment_col,
        )

        predictions = (
            fold_val[segment_col]
            .map(segment_el["expected_loss"])
        )

        # Unseen / missing segment fallback
        predictions = predictions.fillna(
            portfolio_el
        )

        predicted_total = predictions.sum()

        ape = (
            abs(
                predicted_total
                - actual_total
            )
            / actual_total
            * 100
        )

        el_cv_results.append(
            {
                "model": segment_col,
                "fold": fold,
                "predicted_total": predicted_total,
                "actual_total": actual_total,
                "absolute_percentage_error": ape,
            }
        )

el_cv_results = pd.DataFrame(
    el_cv_results
)

In [36]:
el_cv_summary = (
    el_cv_results
    .groupby("model")
    .agg(
        mean_ape=(
            "absolute_percentage_error",
            "mean",
        ),
        median_ape=(
            "absolute_percentage_error",
            "median",
        ),
        std_ape=(
            "absolute_percentage_error",
            "std",
        ),
    )
    .sort_values(
        "mean_ape",
        ascending=True,
    )
)

el_cv_summary

,mean_ape,median_ape,std_ape
model,,,
previous_claims_cat,35.6132,28.6628,21.9888
Portfolio,35.7094,28.4443,22.3160
risk_zone,35.8123,28.8384,22.2441
channel,35.8777,28.8363,22.4529
client_age_group,35.9305,27.6374,21.6847
vehicle_age_group,36.2382,28.6770,22.8718


### Train-Only Segmentation Results

Segment-based Expected Loss approaches do not provide a meaningful improvement over the portfolio benchmark.

`previous_claims_cat` has the lowest mean error, but the improvement is very small.

Risk zone, channel, client age group, and vehicle age group also perform similarly to the simple portfolio approach.

Because the dataset contains a limited number of claims, segment-level frequency and severity estimates are unstable.

For this reason, the simpler portfolio-level Expected Loss benchmark is selected as the final approach.

## Final Expected Loss Model

The final Expected Loss approach uses portfolio-level training estimates.

Individual claim occurrence models did not generalize reliably to the test set.

Pre-claim segmentation also did not provide a meaningful improvement during train-only cross-validation.

Therefore, the final benchmark uses:

- the training portfolio claim rate as claim probability,
- the training mean claim severity as expected severity.

The resulting Expected Loss is approximately 91.8 per Auto contract.

On the untouched test portfolio, the benchmark underestimated the realized total loss by approximately 11.8%.

The Actual-to-Expected ratio was approximately 1.13.

This model should be interpreted as a portfolio-level actuarial benchmark rather than an individual contract risk model.

In [37]:
final_el_summary = pd.Series(
    {
        "train_contracts": len(train_population),
        "train_claims": int(train_population["has_claim"].sum()),
        "train_claim_rate": train_claim_rate,
        "train_mean_severity": train_mean_severity,
        "expected_loss_per_contract": portfolio_expected_loss,

        "test_contracts": len(test_loss_df),
        "test_claims": int(test_loss_df["has_claim"].sum()),

        "predicted_total_loss": predicted_total_loss,
        "actual_total_loss": actual_total_loss,

        "absolute_percentage_error_pct": (
            absolute_percentage_error
        ),

        "actual_to_expected_ratio": (
            loss_ratio_actual_to_expected
        ),
    }
)

final_el_summary

train_contracts                  3,461.0000
train_claims                        70.0000
train_claim_rate                     0.0202
train_mean_severity              4,538.9144
expected_loss_per_contract          91.8012
test_contracts                     866.0000
test_claims                         18.0000
predicted_total_loss            79,499.8534
actual_total_loss               90,158.3300
absolute_percentage_error_pct       11.8220
actual_to_expected_ratio             1.1341
dtype: float64

## Final Conclusion

The final Expected Loss benchmark uses the training portfolio claim rate and the training mean claim severity.

The estimated Expected Loss is approximately 91.8 per Auto contract.

On the untouched test portfolio, the model predicted a total loss of about 79,500 compared with an actual loss of about 90,158.

The absolute percentage error is approximately 11.8%, and the Actual-to-Expected ratio is about 1.13.

More complex segmentation approaches did not provide a meaningful and stable improvement over the portfolio benchmark.

Therefore, the portfolio-level approach is selected as the final Expected Loss methodology for the current dataset.